# Zero-shot fraud labeler

Train `TabularVAE` on normal synthetic card transactions only. Flag fraud by reconstruction error. The hidden fraud label is used for the validation report and never as a training input.

Continuous columns are `log1p`-transformed and standardized with a scaler fit on train normals. Huber loss uses delta 1.0 in that scaled space. The anomaly score ignores KL. The threshold is the 95th percentile of scores on a pure-normal calibration holdout, then frozen. Metrics are computed only on the mixed validation set.

The deployment map for serving this later is `vae-fraud-plan.md` (project plan, not in this repo). This notebook does not start a server.


## Setup

Seed, sizes, and loss weights live in this cell so a rerun is one place to edit. Hidden width, learning rate, batch size, and the exact normal split are local choices. Latent size 6, the loss weights, the 5% fraud rate, the 25 epochs, and `log1p` + standardization match the plan.


In [1]:
from pathlib import Path
import sys

import numpy as np
import torch

cwd = Path.cwd().resolve()
if (cwd / "src" / "fraudvae").is_dir():
    root = cwd
elif (cwd.parent / "src" / "fraudvae").is_dir():
    root = cwd.parent
else:
    raise RuntimeError("Run this notebook from the repo root or from notebooks/.")
src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)

from fraudvae import (
    LossConfig,
    ReconstructionScorer,
    TabularVAE,
    TransactionDataModule,
    VAELoss,
    VAETrainer,
    detection_report,
    generate_synthetic_transactions,
    set_seed,
)
from fraudvae.score import format_score_stats, text_histogram

SEED = 42
N_NORMAL = 7600
N_FRAUD = 400  # 400 / 8000 = 5%
N_TRAIN = 5200
N_CALIBRATION = 1200
N_VAL_NORMAL = 1200
BATCH_SIZE = 128
LATENT_DIM = 6
HIDDEN_DIM = 64
EPOCHS = 25
LR = 1e-3
LAMBDA_CONT = 1.0
LAMBDA_CAT = 1.0
BETA = 0.1
HUBER_DELTA = 1.0
THRESHOLD_PERCENTILE = 95.0

set_seed(SEED)
print(f"torch {torch.__version__}  device cpu  seed {SEED}")
print(f"rows {N_NORMAL + N_FRAUD}  fraud rate {N_FRAUD / (N_NORMAL + N_FRAUD):.1%}")


torch 2.14.1+cu130  device cpu  seed 42
rows 8000  fraud rate 5.0%


## 1. Data processing

`generate_synthetic_transactions` draws positive, right-skewed continuous features and two categoricals.

Normal rows sit on a short manifold: fee tracks amount, rolling 24h spend falls as seconds since the last transaction grow, and device type follows distance (chip and contactless near home, online farther away). Merchant id follows spend. Premium merchants are `m0` and `m1`. `m8` and `m9` stay rare.

Fraud is the minority and is shifted on purpose: larger amount, a fee that does not grow with it, a long gap together with a high rolling spend, a far distance still tagged `chip`, and a rare merchant (`m7`–`m9`). That shift is what lets reconstruction error beat chance. It is a property of this generator, not a rule we would know in production.

Splits, with no shared `row_id`:

- train loader: normal rows only
- calibration: pure normal, never used to fit weights, used only for the threshold
- validation: the remaining normals plus every fraud row


In [2]:
frame = generate_synthetic_transactions(n_normal=N_NORMAL, n_fraud=N_FRAUD, seed=SEED)
data = TransactionDataModule(
    frame,
    n_train=N_TRAIN,
    n_calibration=N_CALIBRATION,
    n_val_normal=N_VAL_NORMAL,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

print(data.summary())
print("continuous features:", data.schema.continuous_features)
print("categorical vocabs:", data.schema.categorical_vocabs)

# The training loader must not contain fraud. Labels are yielded so this
# audit is possible; the trainer never passes them into the loss.
train_labels = data.train_dataset.labels.numpy()
assert np.all(train_labels == 0), "training loader includes fraud rows"
assert set(data.train_frame.row_id).isdisjoint(data.calibration_frame.row_id)
assert set(data.train_frame.row_id).isdisjoint(data.validation_frame.row_id)
assert set(data.calibration_frame.row_id).isdisjoint(data.validation_frame.row_id)
assert data.validation_frame.label.nunique() == 2

display_cols = [
    "amount",
    "fee",
    "rolling_spend_24h",
    "distance_from_home_km",
    "seconds_since_last_txn",
    "merchant_id",
    "device_type",
    "label",
]
print("\nraw feature medians by hidden label")
print(frame.groupby("label")[display_cols[:-3]].median().round(2))

scaled = data.preprocessor.transform(data.validation_frame)
val_labels = data.validation_frame.label.to_numpy()
print("\nscaled validation means (train-normal log1p z-score)")
print("normal", np.round(scaled[val_labels == 0].mean(axis=0), 3))
print("fraud ", np.round(scaled[val_labels == 1].mean(axis=0), 3))
print("scaler mean", np.round(data.preprocessor.mean_, 3))
print("scaler std ", np.round(data.preprocessor.std_, 3))


train: 5200 rows (5200 normal, 0 fraud)
calibration: 1200 rows (1200 normal, 0 fraud)
validation: 1600 rows (1200 normal, 400 fraud)
continuous features: ('amount', 'fee', 'rolling_spend_24h', 'distance_from_home_km', 'seconds_since_last_txn')
categorical vocabs: (('merchant_id', ('m0', 'm1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7', 'm8', 'm9')), ('device_type', ('chip', 'contactless', 'online')))

raw feature medians by hidden label
       amount   fee  rolling_spend_24h  distance_from_home_km  \
label                                                           
0       24.47  0.50              57.11                   3.05   
1      267.91  1.33             898.06                  79.67   

       seconds_since_last_txn  
label                          
0                     6708.03  
1                    71842.00  

scaled validation means (train-normal log1p z-score)
normal [-0.039 -0.044 -0.029  0.003 -0.005]
fraud  [4.936 2.329 4.839 7.938 4.698]
scaler mean [3.246 0.432 4.068 1.42  8.81

## 2. VAE architecture

`TabularVAE` concatenates the 5 scaled continuous values with one-hot merchant and device vectors, in vocab order. The encoder emits `mu` and `logvar` for a latent vector of size 6. Training samples `z` with the standard Gaussian reparameterization. The decoder has one linear head for the continuous columns and one linear head of raw logits per categorical column. There is no softmax on those heads.


In [3]:
model = TabularVAE(
    n_continuous=len(data.schema.continuous_features),
    cat_cardinalities=data.schema.cardinalities,
    latent_dim=LATENT_DIM,
    hidden_dim=HIDDEN_DIM,
)
print(model)
print("latent_dim", model.latent_dim, "cardinalities", model.cat_cardinalities)

# Shape check uses mu, so it does not draw reparameterization noise and
# does not touch the training loader's shuffle order.
model.eval()
with torch.no_grad():
    dummy_continuous = torch.zeros(4, model.n_continuous)
    dummy_categorical = torch.zeros(4, len(model.cat_cardinalities), dtype=torch.long)
    mu, logvar = model.encode(dummy_continuous, dummy_categorical)
    recon_continuous, categorical_logits = model.decode(mu)
print("mu", tuple(mu.shape), "logvar", tuple(logvar.shape))
print("recon continuous", tuple(recon_continuous.shape))
print("categorical logits", [tuple(head.shape) for head in categorical_logits])


TabularVAE(
  (encoder): Sequential(
    (0): Linear(in_features=18, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (fc_mu): Linear(in_features=64, out_features=6, bias=True)
  (fc_logvar): Linear(in_features=64, out_features=6, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=6, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (continuous_head): Linear(in_features=64, out_features=5, bias=True)
  (categorical_heads): ModuleList(
    (0): Linear(in_features=64, out_features=10, bias=True)
    (1): Linear(in_features=64, out_features=3, bias=True)
  )
)
latent_dim 6 cardinalities [10, 3]
mu (4, 6) logvar (4, 6)
recon continuous (4, 5)
categorical logits [(4, 10), (4, 3)]


## 3. Custom loss

`total = lambda_cont * Huber + lambda_cat * cross_entropy + beta * KL`

- Huber is `F.huber_loss` with delta 1.0, mean over the batch and the continuous columns.
- Cross-entropy is `F.cross_entropy` on each head (logits against the class index), mean over the batch, summed across merchant and device.
- KL is the closed form between `q(z|x) = N(mu, diag(exp(logvar)))` and `N(0, I)`, summed over latent dimensions and averaged over the batch.

The epoch log prints those three unweighted pieces plus the weighted total. `beta` is 0.1, so a KL value near 6 contributes about 0.6 to the total, next to the Huber and cross-entropy terms rather than instead of them.


In [4]:
loss_config = LossConfig(
    lambda_cont=LAMBDA_CONT,
    lambda_cat=LAMBDA_CAT,
    beta=BETA,
    huber_delta=HUBER_DELTA,
)
loss_fn = VAELoss(loss_config)
print(loss_config)


LossConfig(lambda_cont=1.0, lambda_cat=1.0, beta=0.1, huber_delta=1.0)


## 4. Training loop

Fit on the normal-only loader. CPU, 25 epochs. The fraud label is not an input.


In [5]:
trainer = VAETrainer(model, loss_fn, lr=LR, device="cpu")
history = trainer.fit(data.train_loader, epochs=EPOCHS)
last = history[-1]
weighted_huber = LAMBDA_CONT * last["huber"]
weighted_ce = LAMBDA_CAT * last["cross_entropy"]
weighted_kl = BETA * last["kl"]
print(
    "last epoch weighted parts  "
    f"huber={weighted_huber:.4f}  "
    f"cross_entropy={weighted_ce:.4f}  "
    f"beta*kl={weighted_kl:.4f}"
)


epoch 01  total=3.6965  huber=0.4220  cross_entropy=3.2698  kl=0.0478
epoch 02  total=3.5043  huber=0.4000  cross_entropy=3.0412  kl=0.6319


epoch 03  total=2.8932  huber=0.3052  cross_entropy=2.2693  kl=3.1866
epoch 04  total=2.3223  huber=0.3267  cross_entropy=1.5677  kl=4.2788


epoch 05  total=1.9149  huber=0.3414  cross_entropy=1.0488  kl=5.2465
epoch 06  total=1.5874  huber=0.3360  cross_entropy=0.6644  kl=5.8704


epoch 07  total=1.3337  huber=0.3159  cross_entropy=0.4142  kl=6.0357
epoch 08  total=1.1739  huber=0.2855  cross_entropy=0.2877  kl=6.0060


epoch 09  total=1.0978  huber=0.2528  cross_entropy=0.2248  kl=6.2019
epoch 10  total=1.0501  huber=0.2369  cross_entropy=0.1864  kl=6.2676


...

epoch 14  total=0.9544  huber=0.2161  cross_entropy=0.1252  kl=6.1305
epoch 15  total=0.9330  huber=0.2160  cross_entropy=0.1157  kl=6.0124


epoch 16  total=0.9294  huber=0.2163  cross_entropy=0.1115  kl=6.0165
epoch 17  total=0.9256  huber=0.2125  cross_entropy=0.1054  kl=6.0766


epoch 18  total=0.9216  huber=0.2115  cross_entropy=0.1016  kl=6.0851
epoch 19  total=0.9155  huber=0.2119  cross_entropy=0.0979  kl=6.0572


epoch 20  total=0.8980  huber=0.2123  cross_entropy=0.0913  kl=5.9441
epoch 21  total=0.8962  huber=0.2122  cross_entropy=0.0893  kl=5.9471
epoch 22  total=0.8958  huber=0.2139  cross_entropy=0.0862  kl=5.9573


epoch 23  total=0.8859  huber=0.2122  cross_entropy=0.0836  kl=5.9009
epoch 24  total=0.8731  huber=0.2119  cross_entropy=0.0833  kl=5.7789
epoch 25  total=0.8791  huber=0.2116  cross_entropy=0.0861  kl=5.8136
last epoch weighted parts  huber=0.2116  cross_entropy=0.0861  beta*kl=0.5814


## 5. Anomaly scoring and thresholding

One score per row, KL left out:

`lambda_cont * huber(continuous) + lambda_cat * (cross_entropy(merchant) + cross_entropy(device))`

Per-row Huber and cross-entropy are sums within the row. Training logged means; the threshold lives in this per-row scale. Decoding uses `z = mu`, not a fresh sample, so the same transaction always gets the same score.

The threshold is the 95th percentile on the calibration normals only. A validation row at or above that value is `High-Risk Anomaly`. The classification report below is validation only. Positive class is fraud.

`score(transaction)` is the same function a later service would call. The hand-written calls at the bottom of this section go through it, including one batch of two.


In [6]:
scorer = ReconstructionScorer(
    model=model,
    preprocessor=data.preprocessor,
    loss_config=loss_config,
    schema=data.schema,
)
threshold = scorer.fit_threshold(data.calibration_frame, percentile=THRESHOLD_PERCENTILE)
print(f"threshold ({THRESHOLD_PERCENTILE:.0f}th pct of calibration normals) = {threshold:.4f}")

scored = scorer.score_frame(data.validation_frame)
labels = data.validation_frame.label.to_numpy()
scores = scored["reconstruction_error"].to_numpy()
high_risk = scored["high_risk"].to_numpy()

calibration_scores = scorer.reconstruction_errors(data.calibration_frame)["reconstruction_error"].to_numpy()
print(format_score_stats("calibration normal", calibration_scores))
print(format_score_stats("validation normal", scores[labels == 0]))
print(format_score_stats("validation fraud", scores[labels == 1]))

print("\nvalidation normal scores")
print(text_histogram(scores[labels == 0]))
print("\nvalidation fraud scores")
print(text_histogram(scores[labels == 1]))

print("\n" + detection_report(labels, high_risk))

# A score with no class signal would flag about 5% of fraud at this
# percentile. That is the chance recall to beat.
flag_rate_normal = float(high_risk[labels == 0].mean())
flag_rate_fraud = float(high_risk[labels == 1].mean())
print(f"flag rate  normal={flag_rate_normal:.3f}  fraud={flag_rate_fraud:.3f}")
print("chance recall at a 95th-percentile threshold is about 0.05 if scores do not depend on class")

examples = data.validation_frame.copy()
examples["reconstruction_error"] = scores
examples["flag"] = scored["flag"].to_numpy()
show = [
    "amount",
    "fee",
    "rolling_spend_24h",
    "distance_from_home_km",
    "seconds_since_last_txn",
    "merchant_id",
    "device_type",
    "label",
    "reconstruction_error",
    "flag",
]
flagged = examples.loc[examples["flag"] == "High-Risk Anomaly"]
print("\nflagged fraud rows (true positives)")
print(flagged.loc[flagged["label"] == 1, show].head(3).round(3).to_string(index=False))
print("\nflagged normal rows (false positives at this threshold)")
print(flagged.loc[flagged["label"] == 0, show].head(3).round(3).to_string(index=False))

# Same contract the service would use. score_batch is score() over a list.
# hand_normal sits on the generator's manifold: fee tracks amount, rolling
# spend matches a gap of about two hours, merchant m3 and contactless are
# ordinary for that spend and distance. hand_suspicious breaks those links.
score = scorer.score
hand_normal = {
    "amount": 24.5,
    "fee": 0.50,
    "rolling_spend_24h": 57.0,
    "distance_from_home_km": 3.0,
    "seconds_since_last_txn": 6600,
    "merchant_id": "m3",
    "device_type": "contactless",
}
hand_suspicious = {
    "amount": 280.0,
    "fee": 1.2,
    "rolling_spend_24h": 900.0,
    "distance_from_home_km": 80.0,
    "seconds_since_last_txn": 80000,
    "merchant_id": "m8",
    "device_type": "chip",
}
print("\nscore(hand_normal)")
normal_result = score(hand_normal)
print(normal_result)
print("score(hand_suspicious)")
suspicious_result = score(hand_suspicious)
print(suspicious_result)
print("score_batch")
print(scorer.score_batch([hand_normal, hand_suspicious]))

# Lambdas are both 1, so the pieces add up to the reconstruction error.
for name, result in (("normal", normal_result), ("suspicious", suspicious_result)):
    parts = result["continuous_error"] + sum(result["categorical_error"].values())
    if abs(parts - result["reconstruction_error"]) > 1e-4:
        raise AssertionError(f"{name} score parts do not add up: {parts} vs {result['reconstruction_error']}")


threshold (95th pct of calibration normals) = 2.1162


calibration normal: n=1200  mean=0.843  std=0.669  p05=0.100  p50=0.694  p95=2.116  min=0.016  max=4.259
validation normal: n=1200  mean=0.818  std=0.640  p05=0.115  p50=0.667  p95=2.040  min=0.013  max=4.037
validation fraud: n=400  mean=23.257  std=3.967  p05=16.521  p50=24.344  p95=28.517  min=13.189  max=30.878

validation normal scores
   0.01 –    0.35 | ######################################## 333
   0.35 –    0.68 | ##################################       281
   0.68 –    1.02 | #########################                207
   1.02 –    1.35 | #####################                    178
   1.35 –    1.69 | ###########                              90
   1.69 –    2.03 | ######                                   48
   2.03 –    2.36 | ###                                      25
   2.36 –    2.70 | ##                                       16
   2.70 –    3.03 | #                                        11
   3.03 –    3.37 | #                                        5
   3.37 –    3

## How to read the report

Fraud here was generated off the normal manifold, several standard deviations out on the scaled continuous columns, with merchant and device pairs the normal process rarely emits. A high F1 on this notebook means the reconstruction score noticed that synthetic shift. It does not say how the same network would do on a real book of transactions.

If a later generator produces a weak F1, the useful response is to revisit the generator or the loss balance, not to treat the number as a production operating point. Unknown merchant ids are rejected by `score` rather than mapped to a zero vector. The threshold stays the calibration number; it is not recomputed on the validation rows.
